# Preprocessing Pipeline

Contains the preprocessing code for thesis by Aurelija Spunde.

Mainly based on tutorials from MNE-Python documentation page (https://mne.tools/stable/auto_tutorials/intro/10_overview.html#preprocessing and other tutorials)

## Loading the data

Code from build.py

In [ ]:
"""

When the real data is available, can run this in the terminal to get the files

uv run build.py --data-root /mnt/FastData/EXPERIMENT_DATA --out out/
uv run build.py --physics        # adds the physics-state table (~17M rows, 266 MB)
uv run build.py --adhoc-session  # a run that is not in sessions.toml (see Curation)
uv run build.py --export-eeg     # also write each aligned hyperscanner file as .fif
"""

Loading the packages and the raw data

In [1]:
import mne
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# Just copy-pasted code from my previous notebook, will need to be adapted to the new data structure

# Loading data

data_folder = Path("/mnt/FastData/EXPERIMENT_DATA") # change according to the actual path of the fif files

# Reading raw data files for each subject

raws = []

for subject in range(1, len(list(data_folder.glob("*.fif")))):
    raw_subject = mne.io.read_raw_fif(data_folder / f'{subject}.fif') # change according to the data structure and file names
    raws.append(raw_subject)
    print(f"Subject: {subject}, Channels: {raw_subject.info['nchan']}, Duration: {raw_subject.times[-1]:.2f} s, Sampling Rate: {raw_subject.info['sfreq']} Hz")
    

# Define montage
montage = mne.channels.make_standard_montage('standard_1010') # look into it

for raw in raws:

    # Set montage
    raw.set_montage(montage)

    raw.set_eeg_reference(ref_channels=['M1', 'M2'])
    
    # Load data
    raw.load_data()

    # Plot
    raw.plot(n_channels=30, scalings=dict(eeg=50e-6))


## ICA on raw data

**Looping over subjects:**

Filtering the data for ICA training

Epoching data and removing artificial epochs for ICA

Running ICA with the runica algorithm and the additional option of PCA

Saving ICA weights to original data


In [ ]:
# Identifying bad channels and interpolating them 
# (in case we need to do this. Maybe it will be done before)

for raw in raws: # Doing it for every subject could be a bit difficult, 
    # but maybe then we should do it without the for loop and just do it 
    # for one subject at a time.

    raw.info.ch_names
    raw.info["bads"] = ["C4", "TP9"]
    raw.interpolate_bads()
    raw.pick(['eeg', 'eog'], exclude='bads')

    # Plotting
    raw.plot(n_channels=30, scalings=scaling)

In [ ]:
# ICA: 1st option

for raw in raws: # add the EMG and IMU

    raw_filt = raw.copy().filter(l_freq=1.0, h_freq=None)
    ica = mne.preprocessing.ICA(n_components=0.95, method="picard", max_iter="auto", rng=97)
    ica.fit(raw_filt, reject=dict(eeg=200e-6))

    ica.plot_components()
    # ica.plot_sources(raw_filt)

    eog_indices, eog_scores = ica.find_bads_eog(raw_filt, ch_name=['EOG2', 'EOG3', 'EOG4']) # Q: is it performed on the original data or the filtered copy? 
    muscle_indices, muscle_scores = ica.find_bads_muscle(raw_filt)

    # ica.exclude = [0, 1, eog_indices, muscle_indices]  # indices chosen based on various plots above - may need to be adjusted for each subject, so maybe we should do it for each subject separately
    ica.exclude = [eog_indices, muscle_indices] # automatic, do not need to be adjusted for each subject

    # ica.plot_overlay(raw, exclude=[0], picks="eeg")
    raw = ica.apply(raw.copy())


# ICA: 2nd option - corrmap

icas = []

for raw in raws: # add the EMG and IMU

    raw_filt = raw.copy().filter(l_freq=1.0, h_freq=None)
    ica = mne.preprocessing.ICA(n_components=0.95, method="picard", max_iter="auto", rng=97)
    ica.fit(raw_filt, reject=dict(eeg=200e-6))
    icas.append(ica)

# use the first subject as template

eog_inds, eog_scores = icas[0].find_bads_eog(raws[0], ch_name=['EOG2', 'EOG3', 'EOG4'])

template_eog_component = icas[0].get_components()[:, eog_inds[0]]
mne.preprocessing.corrmap(icas, template=template_eog_component, threshold='auto', plot=True)
print(template_eog_component)

for index, (ica, raw) in enumerate(zip(icas, raws)):
    with mne.viz.use_browser_backend("matplotlib"):
        fig = ica.plot_sources(raw, show_scrollbars=False)
    fig.subplots_adjust(top=0.9)  # make space for title
    fig.suptitle(f"Subject {index}")

## Preprocessing

**Preprocessing data after ICA cleaning:**

1. Detecting experimental events

2. Filtering the data

3. Epoching

**1. Detecting experimental events**

In [ ]:
events = mne.find_events(raws[0], stim_channel="STI 014") # looking only at the first subject,
# assuming that the events are the same for all subjects, which should be the case
print(events[:5])  # show the first 5

In [ ]:
event_dict = { # adjust to our events
    "individual/balldrop": 1,
    "team/balldrop": 2,
    "visual/left": 3,
    "visual/right": 4,
    "smiley": 5,
    "buttonpress": 32,
}

fig = mne.viz.plot_events(
    events, event_id=event_dict, sfreq=raws[0].info["sfreq"], first_samp=raws[0].first_samp
)

**2. Filtering the data**

In [ ]:
for raw in raws:

    raw.filter(0.1, 40)    

    # Notch filter
    raw.notch_filter(freqs=50)

    # Plotting
    raw.plot(n_channels=30, scalings=100e-6)

**2. Epoching**

In [ ]:
reject_criteria = dict( # adjust to our data
    eeg=150e-6,  # 150 µV
    eog=250e-6 # 250 µV,
)

conditions = ["auditory/left", "auditory/right", "visual/left", "visual/right"]

In [ ]:
for raw in raws:
    
    epochs = mne.Epochs(
        raw,
        events,
        event_id=event_dict,
        tmin=-0.2, # maybe look a bit before the balldrop event to see the anticipatory activity
        tmax=0.5,
        reject=reject_criteria,
        preload=True,
    )

    epochs.equalize_event_counts(conditions)
    aud_epochs = epochs["auditory"]
    vis_epochs = epochs["visual"]

    # Plotting

    aud_epochs.plot_image(picks=["EEG 021"])

    aud_evoked = aud_epochs.average()
    vis_evoked = vis_epochs.average()

    mne.viz.plot_compare_evokeds(
        dict(auditory=aud_evoked, visual=vis_evoked),
        legend="upper left",
        show_sensors="upper right",
    )

    aud_evoked.plot_joint(picks="eeg")

    evoked_diff = mne.combine_evoked([aud_evoked, vis_evoked], weights=[1, -1])
    evoked_diff.pick(picks="mag").plot_topo(color="r", legend=False)




Downsampling to 250 Hz


In [ ]:
for raw in raws:
    epochs = epochs.resample(sfreq=250)
    epochs